# Model 2  Hybrid (RoBERTa + PhoBERT)

File này gồm 3 phần:
1. Train RoBERTa-base trên tiếng Anh  `models/roberta/`
2. Train PhoBERT trên tiếng Việt  `models/phobert/` (segment on-the-fly, không đụng CSV)
3. Kết hợp 2 model với routing logic:
   - `P(lang chính)  0.8`  Hard routing
   - `P(lang chính) < 0.8`  Soft voting (song ngữ)

In [26]:
!pip install transformers scikit-learn torch sentencepiece langdetect underthesea -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 74.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.1 MB/s eta 0:00:00


---
## Phần 1  Train RoBERTa (Tiếng Anh)

In [27]:
import pandas as pd, torch, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
from underthesea import word_tokenize

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

RO_MODEL   = "roberta-base"
RO_OUTPUT  = "models/roberta"
PH_MODEL   = "vinai/phobert-base-v2"
PH_OUTPUT  = "models/phobert"
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5

def segment_vi(text):
    """Word segmentation tiếng Việt cho PhoBERT."""
    return word_tokenize(text, format="text")

class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, segment=False):
        self.texts   = df["text"].tolist()
        self.labels  = df["label"].tolist()
        self.tok     = tokenizer
        self.max_len = max_len
        self.segment = segment  # True chỉ khi dùng PhoBERT
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.segment:
            text = segment_vi(text)  # segment on-the-fly
        enc = self.tok(text, max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label":          torch.tensor(self.labels[idx], dtype=torch.long)}

def train_model(model, train_loader, val_loader, output_dir, tokenizer, epochs=EPOCHS, lr=LR):
    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    best_f1 = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            inputs = {"input_ids":      batch["input_ids"].to(device),
                      "attention_mask": batch["attention_mask"].to(device),
                      "labels":         batch["label"].to(device)}
            out = model(**inputs)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += out.loss.item()
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in val_loader:
                inputs = {"input_ids":      batch["input_ids"].to(device),
                          "attention_mask": batch["attention_mask"].to(device)}
                out = model(**inputs)
                preds.extend(out.logits.argmax(-1).cpu().numpy())
                trues.extend(batch["label"].numpy())
        val_f1 = f1_score(trues, preds, average="macro")
        print(f"  Epoch {epoch+1}/{epochs} | loss: {total_loss/len(train_loader):.4f} | val F1: {val_f1:.4f}")
        if val_f1 > best_f1:
            best_f1 = val_f1
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            print(f"     Saved (F1={best_f1:.4f})")
    return best_f1

Device: cuda


In [28]:
train_df = pd.read_csv("/kaggle/input/datasets/minhbodoi/minh12/processed_data/train.csv", encoding="utf-8-sig")
val_df   = pd.read_csv("/kaggle/input/datasets/minhbodoi/minh12/processed_data/val.csv",   encoding="utf-8-sig")

train_en = train_df[train_df["language"] == "en"].reset_index(drop=True)
val_en   = val_df  [val_df  ["language"] == "en"].reset_index(drop=True)
print(f"[EN] Train: {len(train_en):,}  |  Val: {len(val_en):,}")

tok_ro   = AutoTokenizer.from_pretrained(RO_MODEL)
model_ro = AutoModelForSequenceClassification.from_pretrained(RO_MODEL, num_labels=2).to(device)

train_loader_en = DataLoader(TextDataset(train_en, tok_ro, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader_en   = DataLoader(TextDataset(val_en,   tok_ro, MAX_LEN), batch_size=BATCH_SIZE)

print("\n Training RoBERTa (EN) ")
best_ro = train_model(model_ro, train_loader_en, val_loader_en, RO_OUTPUT, tok_ro)
print(f"\n RoBERTa done. Best val F1: {best_ro:.4f}")

[EN] Train: 12,307  |  Val: 2,637


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



── Training RoBERTa (EN) ──
  Epoch 1/3 | loss: 0.2073 | val F1: 0.9748


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Saved (F1=0.9748)
  Epoch 2/3 | loss: 0.0462 | val F1: 0.9794


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Saved (F1=0.9794)
  Epoch 3/3 | loss: 0.0215 | val F1: 0.9815


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Saved (F1=0.9815)

🏁 RoBERTa done. Best val F1: 0.9815


---
## Phần 2  Train PhoBERT (Tiếng Việt)
Text được segment on-the-fly trong DataLoader  không lưu lại CSV, không ảnh hưởng Model 1.

In [29]:
train_vi = train_df[train_df["language"] == "vi"].reset_index(drop=True)
val_vi   = val_df  [val_df  ["language"] == "vi"].reset_index(drop=True)
print(f"[VI] Train: {len(train_vi):,}  |  Val: {len(val_vi):,}")
print(" DataLoader sẽ segment on-the-fly  train chậm hơn RoBERTa một chút, bình thường.")

tok_ph   = AutoTokenizer.from_pretrained(PH_MODEL)
model_ph = AutoModelForSequenceClassification.from_pretrained(PH_MODEL, num_labels=2).to(device)

# segment=True  on-the-fly word segmentation
train_loader_vi = DataLoader(TextDataset(train_vi, tok_ph, MAX_LEN, segment=True), batch_size=BATCH_SIZE, shuffle=True)
val_loader_vi   = DataLoader(TextDataset(val_vi,   tok_ph, MAX_LEN, segment=True), batch_size=BATCH_SIZE)

print("\n Training PhoBERT (VI) ")
best_ph = train_model(model_ph, train_loader_vi, val_loader_vi, PH_OUTPUT, tok_ph)
print(f"\n PhoBERT done. Best val F1: {best_ph:.4f}")

[VI] Train: 15,557  |  Val: 3,334
⏳ DataLoader sẽ segment on-the-fly — train chậm hơn RoBERTa một chút, bình thường.


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]


── Training PhoBERT (VI) ──
  Epoch 1/3 | loss: 0.1179 | val F1: 0.9935


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Saved (F1=0.9935)
  Epoch 2/3 | loss: 0.0149 | val F1: 0.9910
  Epoch 3/3 | loss: 0.0061 | val F1: 0.9910

🏁 PhoBERT done. Best val F1: 0.9935


---
## Phần 3  Hybrid Inference & Đánh giá

In [30]:
import torch.nn.functional as F
from langdetect import detect_langs, DetectorFactory, LangDetectException
DetectorFactory.seed = 42

LANG_THRESHOLD = 0.8

model_ro_best = AutoModelForSequenceClassification.from_pretrained(RO_OUTPUT).to(device)
tok_ro_best   = AutoTokenizer.from_pretrained(RO_OUTPUT)
model_ph_best = AutoModelForSequenceClassification.from_pretrained(PH_OUTPUT).to(device)
tok_ph_best   = AutoTokenizer.from_pretrained(PH_OUTPUT)
model_ro_best.eval()
model_ph_best.eval()

def detect_language(text):
    try:
        results = detect_langs(text[:500])
        top  = results[0]
        lang = top.lang if top.lang in ["en", "vi"] else "other"
        return lang, top.prob
    except LangDetectException:
        return "other", 0.0

def predict_proba(text, model, tokenizer, segment=False):
    if segment:
        text = segment_vi(text)
    enc = tokenizer(text, max_length=MAX_LEN, padding="max_length",
                    truncation=True, return_tensors="pt")
    inputs = {"input_ids":      enc["input_ids"].to(device),
              "attention_mask": enc["attention_mask"].to(device)}
    with torch.no_grad():
        logits = model(**inputs).logits
    return F.softmax(logits, dim=-1)[0][1].item()

def hybrid_predict(text):
    lang, lang_prob = detect_language(text)
    if lang_prob >= LANG_THRESHOLD:
        if lang == "en":
            p_ai     = predict_proba(text, model_ro_best, tok_ro_best)
            strategy = "hardroberta"
        elif lang == "vi":
            p_ai     = predict_proba(text, model_ph_best, tok_ph_best, segment=True)
            strategy = "hardphobert"
        else:
            p_en = predict_proba(text, model_ro_best, tok_ro_best)
            p_vi = predict_proba(text, model_ph_best, tok_ph_best, segment=True)
            p_ai     = (p_en + p_vi) / 2
            strategy = "softother"
    else:
        p_en = predict_proba(text, model_ro_best, tok_ro_best)
        p_vi = predict_proba(text, model_ph_best, tok_ph_best, segment=True)
        p_ai     = (p_en + p_vi) / 2
        strategy = "softbilingual"
    return (1 if p_ai >= 0.5 else 0), p_ai, strategy

print(" Hybrid model ready")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Hybrid model ready


In [31]:
from tqdm import tqdm

test_df = pd.read_csv("/kaggle/input/datasets/minhbodoi/minh12/processed_data/test.csv", encoding="utf-8-sig")
print(f"Test set: {len(test_df):,} records")

preds, strategies = [], []
for text in tqdm(test_df["text"].tolist(), desc="Predicting"):
    label, p_ai, strategy = hybrid_predict(text)
    preds.append(label)
    strategies.append(strategy)

test_df["pred"]     = preds
test_df["strategy"] = strategies

print("\n Hybrid  Tổng:")
print(classification_report(test_df["label"], test_df["pred"], target_names=["Human", "AI"]))

for lang in ["en", "vi"]:
    sub = test_df[test_df["language"] == lang]
    print(f"\n {lang.upper()} ")
    print(classification_report(sub["label"], sub["pred"], target_names=["Human", "AI"]))

print("\n Routing strategy ")
print(test_df["strategy"].value_counts().to_string())

Test set: 5,972 records


Predicting: 100%|██████████| 5972/5972 [01:43<00:00, 57.64it/s]



📊 Hybrid — Tổng:
              precision    recall  f1-score   support

       Human       1.00      0.98      0.99      3034
          AI       0.98      1.00      0.99      2938

    accuracy                           0.99      5972
   macro avg       0.99      0.99      0.99      5972
weighted avg       0.99      0.99      0.99      5972


── EN ──
              precision    recall  f1-score   support

       Human       0.99      0.97      0.98      1074
          AI       0.98      0.99      0.99      1564

    accuracy                           0.98      2638
   macro avg       0.99      0.98      0.98      2638
weighted avg       0.98      0.98      0.98      2638


── VI ──
              precision    recall  f1-score   support

       Human       1.00      0.99      0.99      1960
          AI       0.98      1.00      0.99      1374

    accuracy                           0.99      3334
   macro avg       0.99      0.99      0.99      3334
weighted avg       0.99      0.99   